# Inverse Positioning From Magnetic Trajectories

This notebook builds an Earth-scale synthetic magnetic-like field on sphere coordinates `x = lon`, `y = sin(lat)`, simulates noisy biased-direction random-walk measurement trajectories with 1 m geodesic steps, and estimates `H(end | trajectory)` and `I(end; trajectory)` from a discrete uniform endpoint prior. Endpoint likelihoods are scored directly from observed `lon,y` coordinate displacements.


## 1. Preparations

In [1]:
import math
from dataclasses import dataclass

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.special import logsumexp
from tqdm.auto import tqdm

SEED = 42

# Earth-scale geometry and magnetic-observation controls.
EARTH_RADIUS_M = 6_371_000.0
STEP_METERS = 500.0

# NOISE is the absolute Gaussian std used for augmented field measurements and
# for the trajectory observation model.
NOISE = 3e-3

# Trajectory experiment controls.
STEP_SIZE = STEP_METERS / EARTH_RADIUS_M
STEP_COUNT_CHOICES = np.array([1, 2, 4, 8, 12, 16], dtype=int)
EXPERIMENT_REPEATS = 10

# Biased-direction random walk controls.
DIRECTION_HEADING = 0.0
DIRECTION_MAGNITUDE = 0.75
DIRECTION_MAX_KAPPA = 12.0

# Posterior approximation controls.
GRID_LON_BINS = 128
GRID_Y_BINS = int(math.pi * GRID_LON_BINS // 2)
POSTERIOR_BATCH = 4096

# Smooth internal magnetic dipoles. Sources stay inside the sphere so the field
# is smooth on the surface and periodic in longitude.
FIELD_SOURCE_POSITIONS = np.array(
    [
        [0.00, 0.00, 0.00],
        [0.32, -0.08, 0.10],
        [-0.18, 0.26, -0.22],
        [0.09, 0.05, 0.34],
    ],
    dtype=float,
)
FIELD_SOURCE_MOMENTS = np.array(
    [
        [0.20, -0.10, 1.00],
        [-0.70, 0.40, 0.30],
        [0.50, 0.80, -0.20],
        [0.30, -0.60, 0.50],
    ],
    dtype=float,
)
FIELD_SOURCE_WEIGHTS = np.array([1.00, 0.22, 0.18, 0.14], dtype=float)
FIELD_EPS = 1e-12

# Earth-scale detail controls. These plane waves are evaluated from Cartesian
# chord projections measured in meters, then restricted to the sphere. The
# shortest wavelengths are intentionally meter-scale so sub-kilometer walks are visible.
FIELD_DETAIL_SEED = 2025
FIELD_DETAIL_OCTAVES = 8
FIELD_DETAIL_BASIS_PER_OCTAVE = 12
FIELD_DETAIL_SCALES_M = np.array(
    [50_000.0, 10_000.0, 2_000.0, 400.0, 80.0, 20.0, 5.0, 1.25], dtype=float
)
FIELD_DETAIL_AMPLITUDES = np.array(
    [0.12, 0.10, 0.08, 0.06, 0.045, 0.035, 0.028, 0.022], dtype=float
)

field_detail_rng = np.random.default_rng(FIELD_DETAIL_SEED)
FIELD_DETAIL_DIRECTIONS = field_detail_rng.normal(
    size=(FIELD_DETAIL_OCTAVES, FIELD_DETAIL_BASIS_PER_OCTAVE, 3)
)
FIELD_DETAIL_DIRECTIONS /= np.linalg.norm(
    FIELD_DETAIL_DIRECTIONS, axis=-1, keepdims=True
)
FIELD_DETAIL_SIN_VECTORS = field_detail_rng.normal(
    size=(FIELD_DETAIL_OCTAVES, FIELD_DETAIL_BASIS_PER_OCTAVE, 3)
)
FIELD_DETAIL_COS_VECTORS = field_detail_rng.normal(
    size=(FIELD_DETAIL_OCTAVES, FIELD_DETAIL_BASIS_PER_OCTAVE, 3)
)
FIELD_DETAIL_PHASES = field_detail_rng.uniform(
    0.0, 2.0 * np.pi, size=(FIELD_DETAIL_OCTAVES, FIELD_DETAIL_BASIS_PER_OCTAVE)
)
FIELD_DETAIL_FREQUENCIES = 2.0 * np.pi / FIELD_DETAIL_SCALES_M

rng = np.random.default_rng(SEED)
np.set_printoptions(precision=4, suppress=True)

c:\Users\shich\Src\inverse_positioning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Shared Functional

Sphere geometry, smooth field generation, dataset creation, trajectory stepping, posterior entropy estimation, and Plotly helpers shared by all experiments.

### Smooth sphere field

The field is evaluated from dipole sources plus Earth-scale plane-wave detail in 3D Cartesian coordinates and then sampled on the unit sphere. This makes longitude wrapping exact because `x = -pi` and `x = pi` map to the same points. It also avoids singular sources on the surface. The shortest detail scale is meter-scale so 1 m walks create observable magnetic differences.

In [2]:
def wrap_lon(lon):
    """Wrap longitude to [-pi, pi)."""
    return (np.asarray(lon) + np.pi) % (2.0 * np.pi) - np.pi


def sphere_xyz(lon, y):
    """Convert lon and y=sin(lat) to unit-sphere Cartesian coordinates."""
    lon, y = np.broadcast_arrays(
        np.asarray(lon, dtype=float), np.asarray(y, dtype=float)
    )
    y = np.clip(y, -1.0, 1.0)
    rho = np.sqrt(np.maximum(0.0, 1.0 - y * y))
    return np.stack((rho * np.cos(lon), rho * np.sin(lon), y), axis=-1)


def lon_y_from_xyz(xyz):
    """Convert unit-sphere Cartesian coordinates back to lon and y=sin(lat)."""
    xyz = np.asarray(xyz, dtype=float)
    lon = np.arctan2(xyz[..., 1], xyz[..., 0])
    y = np.clip(xyz[..., 2], -1.0, 1.0)
    return wrap_lon(lon), y


def sphere_fractal_detail(lon, y):
    """Smooth vector-valued octave noise restricted to the unit sphere."""
    r = sphere_xyz(lon, y)
    detail = np.zeros_like(r)

    for frequency, amplitude, directions, sin_vectors, cos_vectors, phases in zip(
        FIELD_DETAIL_FREQUENCIES,
        FIELD_DETAIL_AMPLITUDES,
        FIELD_DETAIL_DIRECTIONS,
        FIELD_DETAIL_SIN_VECTORS,
        FIELD_DETAIL_COS_VECTORS,
        FIELD_DETAIL_PHASES,
    ):
        projected_m = EARTH_RADIUS_M * np.einsum("...d,bd->...b", r, directions)
        argument = frequency * projected_m + phases
        octave = np.einsum("...b,bd->...d", np.sin(argument), sin_vectors) + np.einsum(
            "...b,bd->...d", np.cos(argument), cos_vectors
        )
        detail += amplitude * octave / math.sqrt(FIELD_DETAIL_BASIS_PER_OCTAVE)

    return detail


def magnetic_field(lon, y):
    """Magnetic-like vector field B(lon, y) = (Bx, By, Bz)."""
    r = sphere_xyz(lon, y)
    field = np.zeros_like(r)

    for source_pos, moment, weight in zip(
        FIELD_SOURCE_POSITIONS, FIELD_SOURCE_MOMENTS, FIELD_SOURCE_WEIGHTS
    ):
        R = r - source_pos
        radius2 = np.sum(R * R, axis=-1, keepdims=True)
        radius = np.sqrt(np.maximum(radius2, FIELD_EPS))
        moment_dot_R = np.sum(moment * R, axis=-1, keepdims=True)
        field += weight * (3.0 * R * moment_dot_R / radius**5 - moment / radius**3)

    # The coordinate-like backbone keeps the global inverse problem identifiable,
    # while the Earth-scale detail supplies local variation at meter-scale steps.
    return 0.70 * r + 0.12 * field + sphere_fractal_detail(lon, y)


def continuity_checks():
    y_probe = np.linspace(-1.0, 1.0, 1_001)
    x_probe = np.linspace(-np.pi, np.pi, 721)
    h = 1e-5

    seam_value_gap = np.max(
        np.abs(magnetic_field(-np.pi, y_probe) - magnetic_field(np.pi, y_probe))
    )
    ddx_left = (
        magnetic_field(-np.pi + h, y_probe) - magnetic_field(-np.pi - h, y_probe)
    ) / (2.0 * h)
    ddx_right = (
        magnetic_field(np.pi + h, y_probe) - magnetic_field(np.pi - h, y_probe)
    ) / (2.0 * h)
    seam_differential_gap = np.max(np.abs(ddx_left - ddx_right))

    north_values = magnetic_field(x_probe, np.ones_like(x_probe))
    south_values = magnetic_field(x_probe, -np.ones_like(x_probe))
    north_d_dx = (
        magnetic_field(x_probe + h, 1.0) - magnetic_field(x_probe - h, 1.0)
    ) / (2.0 * h)
    south_d_dx = (
        magnetic_field(x_probe + h, -1.0) - magnetic_field(x_probe - h, -1.0)
    ) / (2.0 * h)

    return {
        "longitude seam value gap": seam_value_gap,
        "longitude seam d/dx gap": seam_differential_gap,
        "north pole longitude variation": np.ptp(north_values, axis=0).max(),
        "south pole longitude variation": np.ptp(south_values, axis=0).max(),
        "north pole max |dB/dx|": np.linalg.norm(north_d_dx, axis=-1).max(),
        "south pole max |dB/dx|": np.linalg.norm(south_d_dx, axis=-1).max(),
    }


for name, value in continuity_checks().items():
    print(f"{name:34s}: {value:.3e}")

longitude seam value gap          : 3.719e-10
longitude seam d/dx gap           : 2.788e-05
north pole longitude variation    : 0.000e+00
south pole longitude variation    : 0.000e+00
north pole max |dB/dx|            : 0.000e+00
south pole max |dB/dx|            : 0.000e+00


In [3]:
grid_x = np.linspace(-np.pi, np.pi, 241)
grid_y = np.linspace(-1.0, 1.0, 121)
Xg, Yg = np.meshgrid(grid_x, grid_y)
Bg = magnetic_field(Xg, Yg)

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=["Bx", "By", "Bz"],
    horizontal_spacing=0.12,
)

colorbar_x = [0.275, 0.635, 0.995]
for col, component in enumerate(["Bx", "By", "Bz"], start=1):
    fig.add_trace(
        go.Heatmap(
            x=grid_x,
            y=grid_y,
            z=Bg[..., col - 1],
            colorscale="RdBu",
            reversescale=True,
            colorbar={
                "title": component,
                "len": 0.78,
                "thickness": 14,
                "x": colorbar_x[col - 1],
            },
            hovertemplate="x=%{x:.3f}<br>y=%{y:.3f}<br>"
            + component
            + "=%{z:.3f}<extra></extra>",
        ),
        row=1,
        col=col,
    )
    fig.update_xaxes(title_text="x = lon", row=1, col=col, constrain="domain")
    fig.update_yaxes(
        title_text="y = sin(lat)",
        row=1,
        col=col,
        scaleanchor=f"x{col}" if col > 1 else "x",
        scaleratio=np.pi,
        title_standoff=8,
    )

fig.update_layout(
    title={"text": "smooth magnetic field components", "x": 0.5},
    height=440,
    width=1320,
    margin={"l": 60, "r": 90, "t": 70, "b": 70},
)
fig.show()

### Shared trajectory and entropy utilities

In [4]:
def local_east_north(lon, y):
    lon, y = np.broadcast_arrays(
        np.asarray(lon, dtype=float), np.asarray(y, dtype=float)
    )
    rho = np.sqrt(np.maximum(0.0, 1.0 - np.clip(y, -1.0, 1.0) ** 2))
    east = np.stack((-np.sin(lon), np.cos(lon), np.zeros_like(lon)), axis=-1)
    north = np.stack((-y * np.cos(lon), -y * np.sin(lon), rho), axis=-1)
    return east, north


def geodesic_step(lon, y, step_size, heading):
    r = sphere_xyz(lon, y)
    east, north = local_east_north(lon, y)
    heading = np.asarray(heading, dtype=float)
    tangent = (
        np.cos(heading)[..., np.newaxis] * east
        + np.sin(heading)[..., np.newaxis] * north
    )
    r_next = np.cos(step_size) * r + np.sin(step_size) * tangent
    r_next = r_next / np.linalg.norm(r_next, axis=-1, keepdims=True)
    return lon_y_from_xyz(r_next)


def apply_walk(start_lon, start_y, step_size, headings):
    lon = np.asarray(start_lon, dtype=float)
    y = np.asarray(start_y, dtype=float)
    lons = [lon]
    ys = [y]

    for heading in headings:
        lon, y = geodesic_step(lon, y, step_size, heading)
        lons.append(lon)
        ys.append(y)

    return np.stack(lons, axis=-1), np.stack(ys, axis=-1)


def trajectory_displacements(positions_lon, positions_y):
    positions_lon = np.asarray(positions_lon, dtype=float)
    positions_y = np.asarray(positions_y, dtype=float)
    dlon = wrap_lon(np.diff(positions_lon))
    dy = np.diff(positions_y)
    return np.stack([dlon, dy], axis=-1)


def reconstruct_positions_from_endpoint(end_lon, end_y, displacements):
    end_lon = np.asarray(end_lon, dtype=float).reshape(-1)
    end_y = np.asarray(end_y, dtype=float).reshape(-1)
    displacements = np.asarray(displacements, dtype=float)
    step_count = displacements.shape[0]
    count = end_lon.size

    lons = np.empty((count, step_count + 1), dtype=float)
    ys = np.empty((count, step_count + 1), dtype=float)
    lons[:, -1] = wrap_lon(end_lon)
    ys[:, -1] = end_y

    for step_index in range(step_count - 1, -1, -1):
        lons[:, step_index] = wrap_lon(
            lons[:, step_index + 1] - displacements[step_index, 0]
        )
        ys[:, step_index] = ys[:, step_index + 1] - displacements[step_index, 1]

    valid = np.all((ys >= -1.0) & (ys <= 1.0), axis=1)
    return lons, ys, valid


def predicted_field_from_endpoint(end_lon, end_y, displacements):
    lons, ys, valid = reconstruct_positions_from_endpoint(end_lon, end_y, displacements)
    predicted_B = magnetic_field(lons, np.clip(ys, -1.0, 1.0))
    return predicted_B, valid


@dataclass(frozen=True)
class Trajectory:
    repeat: int
    start_lon: float
    start_y: float
    step_size: float
    requested_step_count: int
    headings: np.ndarray
    positions_lon: np.ndarray
    positions_y: np.ndarray
    clean_B: np.ndarray
    observed_B: np.ndarray

    @property
    def step_count(self):
        return len(self.headings)


def make_headings(rng, step_count):
    kappa = DIRECTION_MAGNITUDE * DIRECTION_MAX_KAPPA
    return rng.vonmises(DIRECTION_HEADING, kappa, size=step_count)


def make_trajectory(rng, step_size, step_count, repeat):
    start_lon = rng.uniform(-np.pi, np.pi)
    start_y = rng.uniform(-1.0, 1.0)
    headings = make_headings(rng, step_count)

    positions_lon, positions_y = apply_walk(start_lon, start_y, step_size, headings)
    clean_B = magnetic_field(positions_lon, positions_y)
    observed_B = clean_B + rng.normal(0.0, NOISE, size=clean_B.shape)

    return Trajectory(
        repeat=repeat,
        start_lon=start_lon,
        start_y=start_y,
        step_size=step_size,
        requested_step_count=step_count,
        headings=headings,
        positions_lon=positions_lon,
        positions_y=positions_y,
        clean_B=clean_B,
        observed_B=observed_B,
    )


def make_trajectories(rng):
    return [
        make_trajectory(rng, STEP_SIZE, int(step_count), repeat)
        for step_count in STEP_COUNT_CHOICES
        for repeat in range(EXPERIMENT_REPEATS)
    ]

### Conditional entropy and mutual information

The endpoint prior is represented by a uniform grid directly in endpoint coordinates. For each observed trajectory, the coordinate displacement sequence reconstructs the candidate path backward from each candidate endpoint. The Gaussian magnetic residual likelihood then gives posterior mass over endpoints without searching over candidate starts.

In [5]:
def candidate_grid(n_lon, n_y):
    dx = 2.0 * np.pi / n_lon
    dy = 2.0 / n_y
    lon_centers = np.linspace(-np.pi + 0.5 * dx, np.pi - 0.5 * dx, n_lon)
    y_centers = np.linspace(-1.0 + 0.5 * dy, 1.0 - 0.5 * dy, n_y)
    L, Y = np.meshgrid(lon_centers, y_centers, indexing="xy")
    return L.ravel(), Y.ravel()


def angular_distance(lon1, y1, lon2, y2):
    r1 = sphere_xyz(lon1, y1)
    r2 = sphere_xyz(lon2, y2)
    cosine = np.sum(r1 * r2, axis=-1)
    return np.arccos(np.clip(cosine, -1.0, 1.0))


CANDIDATE_LON, CANDIDATE_Y = candidate_grid(GRID_LON_BINS, GRID_Y_BINS)
CANDIDATES = CANDIDATE_LON.size
PRIOR_ENTROPY_NATS = math.log(CANDIDATES)
PRIOR_ENTROPY_BITS = PRIOR_ENTROPY_NATS / math.log(2.0)

print(f"candidate endpoint states: {CANDIDATES:,}")
print(
    f"discrete endpoint prior entropy: {PRIOR_ENTROPY_NATS:.3f} nats = {PRIOR_ENTROPY_BITS:.3f} bits"
)

candidate endpoint states: 25,728
discrete endpoint prior entropy: 10.155 nats = 14.651 bits


In [6]:
def trajectory_log_likelihood(trajectory):
    displacements = trajectory_displacements(
        trajectory.positions_lon, trajectory.positions_y
    )
    log_likelihood = np.full(CANDIDATES, -np.inf, dtype=float)

    for start in range(0, CANDIDATES, POSTERIOR_BATCH):
        stop = min(start + POSTERIOR_BATCH, CANDIDATES)
        cand_end_lon = CANDIDATE_LON[start:stop]
        cand_end_y = CANDIDATE_Y[start:stop]

        pred_B, valid = predicted_field_from_endpoint(
            cand_end_lon, cand_end_y, displacements
        )
        if not np.any(valid):
            continue
        residual = pred_B[valid] - trajectory.observed_B[np.newaxis, :, :]
        batch_log_likelihood = np.full(stop - start, -np.inf, dtype=float)
        batch_log_likelihood[valid] = (
            -0.5 * np.sum(residual * residual, axis=(1, 2)) / (NOISE * NOISE)
        )
        log_likelihood[start:stop] = batch_log_likelihood

    return log_likelihood


def posterior_for_trajectory(trajectory):
    log_likelihood = trajectory_log_likelihood(trajectory)
    finite = np.isfinite(log_likelihood)
    log_posterior = np.full_like(log_likelihood, -np.inf)
    log_posterior[finite] = log_likelihood[finite] - logsumexp(log_likelihood[finite])
    endpoint_posterior = np.zeros_like(log_likelihood)
    endpoint_posterior[finite] = np.exp(log_posterior[finite])
    entropy_nats = -np.sum(endpoint_posterior[finite] * log_posterior[finite])
    mi_nats = PRIOR_ENTROPY_NATS - entropy_nats

    map_index = int(np.argmax(endpoint_posterior))
    map_end_lon = CANDIDATE_LON[map_index]
    map_end_y = CANDIDATE_Y[map_index]
    true_end_lon = trajectory.positions_lon[-1]
    true_end_y = trajectory.positions_y[-1]

    return {
        "posterior": endpoint_posterior,
        "entropy_nats": entropy_nats,
        "entropy_bits": entropy_nats / math.log(2.0),
        "mi_nats": mi_nats,
        "mi_bits": mi_nats / math.log(2.0),
        "map_end_lon": map_end_lon,
        "map_end_y": map_end_y,
        "map_end_error_rad": float(
            angular_distance(true_end_lon, true_end_y, map_end_lon, map_end_y)
        ),
        "posterior_perplexity": float(np.exp(entropy_nats)),
        "valid_endpoint_candidates": int(np.sum(finite)),
    }


def estimate_information(trajectories, progress_label="trajectory"):
    rows = []
    iterator = tqdm(
        enumerate(trajectories),
        total=len(trajectories),
        desc=f"estimating {progress_label}",
        unit="traj",
    )
    for index, trajectory in iterator:
        posterior = posterior_for_trajectory(trajectory)
        rows.append(
            {
                "trajectory": index,
                "repeat": trajectory.repeat,
                "step_size": trajectory.step_size,
                "steps": trajectory.step_count,
                "conditional_end_entropy_nats": posterior["entropy_nats"],
                "conditional_end_entropy_bits": posterior["entropy_bits"],
                "end_mutual_information_nats": posterior["mi_nats"],
                "end_mutual_information_bits": posterior["mi_bits"],
                "posterior_perplexity": posterior["posterior_perplexity"],
                "map_end_error_rad": posterior["map_end_error_rad"],
                "valid_endpoint_candidates": posterior["valid_endpoint_candidates"],
            }
        )
    return rows


def summarize_values(values):
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        return np.nan, 0.0, 0
    sem = values.std(ddof=1) / np.sqrt(values.size) if values.size > 1 else 0.0
    return values.mean(), sem, values.size


def rows_to_arrays(rows):
    return {
        "conditional_bits": np.array(
            [row["conditional_end_entropy_bits"] for row in rows]
        ),
        "mi_bits": np.array([row["end_mutual_information_bits"] for row in rows]),
        "map_errors": np.array([row["map_end_error_rad"] for row in rows]),
    }


def print_information_summary(rows, title):
    arrays = rows_to_arrays(rows)
    print(f"\n{title}")
    print(f"step size:                     {STEP_METERS:8.4g} m = {STEP_SIZE:8.4g} rad")
    print(f"prior entropy:                 {PRIOR_ENTROPY_BITS:8.3f} bits")
    print(
        f"mean H(end | trajectory):      {arrays['conditional_bits'].mean():8.3f} bits"
    )
    print(
        f"median H(end | trajectory):  {np.median(arrays['conditional_bits']):8.3f} bits"
    )
    print(f"mean I(end; trajectory):        {arrays['mi_bits'].mean():8.3f} bits")
    print(f"median I(end; trajectory):    {np.median(arrays['mi_bits']):8.3f} bits")
    print(f"median endpoint MAP error:      {np.median(arrays['map_errors']):8.4f} rad")


def mi_values(rows, predicate):
    return [row["end_mutual_information_bits"] for row in rows if predicate(row)]


def grouped_mean_sem(rows, group_key, groups):
    output = []
    for group in groups:
        values = [
            row["end_mutual_information_bits"]
            for row in rows
            if row[group_key] == group
        ]
        mean, sem, count = summarize_values(values)
        output.append({"group": group, "mean": mean, "sem": sem, "count": count})
    return output


def add_entropy_line(fig, row=None, col=None, label="H(end)"):
    fig.add_hline(
        y=PRIOR_ENTROPY_BITS,
        line_color="red",
        line_dash="dash",
        line_width=1.2,
        annotation_text=label,
        annotation_position="top left",
        row=row,
        col=col,
    )


def plot_mi_by_steps(rows, title_prefix):
    by_step_count = grouped_mean_sem(rows, "steps", STEP_COUNT_CHOICES)

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=[r["group"] for r in by_step_count],
            y=[r["mean"] for r in by_step_count],
            error_y={
                "type": "data",
                "array": [r["sem"] for r in by_step_count],
                "visible": True,
            },
            mode="lines+markers+text",
            text=[f"n={r['count']}" for r in by_step_count],
            textposition="top center",
            name="steps count",
        )
    )
    add_entropy_line(fig)
    fig.update_xaxes(title_text="steps count", dtick=1)
    fig.update_yaxes(title_text="expected endpoint mutual information [bits]")
    fig.update_layout(
        title=f"{title_prefix}: I(end; trajectory) by steps count, step size={STEP_METERS:g} m",
        height=460,
        width=900,
        margin={"l": 75, "r": 40, "t": 75, "b": 75},
    )
    fig.show()


def wrap_aware_segments(lons, ys, wrap_threshold=np.pi):
    lons = np.asarray(lons, dtype=float)
    ys = np.asarray(ys, dtype=float)
    segments = []
    start = 0
    jumps = np.where(np.abs(np.diff(lons)) > wrap_threshold)[0]
    for jump in jumps:
        stop = jump + 1
        segments.append((lons[start:stop], ys[start:stop]))
        start = stop
    segments.append((lons[start:], ys[start:]))
    return segments


def plot_trajectory_examples(trajectories, title, n_examples=6, seed=SEED + 17):
    rng_examples = np.random.default_rng(seed)
    if len(trajectories) <= n_examples:
        examples = list(trajectories)
    else:
        indices = rng_examples.choice(len(trajectories), size=n_examples, replace=False)
        examples = [trajectories[int(index)] for index in indices]

    cols = min(3, len(examples))
    rows = int(math.ceil(len(examples) / cols))
    fig = make_subplots(
        rows=rows,
        cols=cols,
        subplot_titles=[f"steps={t.step_count}, repeat={t.repeat}" for t in examples],
        horizontal_spacing=0.13,
        vertical_spacing=0.16,
    )

    for index, example in enumerate(examples):
        row = index // cols + 1
        col = index % cols + 1
        for segment_lons, segment_ys in wrap_aware_segments(
            example.positions_lon, example.positions_y
        ):
            fig.add_trace(
                go.Scatter(
                    x=segment_lons,
                    y=segment_ys,
                    mode="lines+markers",
                    line={"width": 2},
                    marker={"size": 6},
                    showlegend=False,
                ),
                row=row,
                col=col,
            )
        fig.add_trace(
            go.Scatter(
                x=[example.start_lon],
                y=[example.start_y],
                mode="markers",
                marker={"symbol": "circle-open", "size": 10, "color": "gray"},
                name="start",
                showlegend=index == 0,
            ),
            row=row,
            col=col,
        )
        fig.add_trace(
            go.Scatter(
                x=[example.positions_lon[-1]],
                y=[example.positions_y[-1]],
                mode="markers",
                marker={"symbol": "star", "size": 13, "color": "red"},
                name="end",
                showlegend=index == 0,
            ),
            row=row,
            col=col,
        )
        fig.update_xaxes(range=[-np.pi, np.pi], title_text="x = lon", row=row, col=col)
        fig.update_yaxes(
            range=[-1.0, 1.0], title_text="y", title_standoff=10, row=row, col=col
        )

    fig.update_layout(
        title={
            "text": f"{title}, step size={STEP_METERS:g} m (gray circle = start, red star = end)",
            "x": 0.5,
        },
        height=350 * rows,
        width=1250,
        margin={"l": 75, "r": 45, "t": 90, "b": 75},
    )
    fig.show()

## 3. Biased Direction Random Walk

This section investigates a single biased-direction random walk. Headings are drawn from a von Mises distribution around `DIRECTION_HEADING`; `DIRECTION_MAGNITUDE` controls how strongly headings concentrate around that direction.

In [7]:
direction_trajectories = make_trajectories(rng)

print(f"biased-direction trajectories: {len(direction_trajectories)}")
print(f"STEP_METERS: {STEP_METERS:g}")
print(f"STEP_SIZE: {STEP_SIZE:g} rad")
print(f"DIRECTION_MAGNITUDE: {DIRECTION_MAGNITUDE:g}")
print(f"DIRECTION_HEADING: {DIRECTION_HEADING:g}")

biased-direction trajectories: 60
STEP_METERS: 500
STEP_SIZE: 7.84806e-05 rad
DIRECTION_MAGNITUDE: 0.75
DIRECTION_HEADING: 0


In [8]:
plot_trajectory_examples(
    direction_trajectories,
    "biased-direction random trajectory examples",
    n_examples=6,
)
direction_information_rows = estimate_information(
    direction_trajectories, progress_label="biased-direction trajectory"
)
print_information_summary(
    direction_information_rows, "Biased-direction endpoint information summary"
)

mean, sem, count = summarize_values(
    mi_values(direction_information_rows, lambda row: True)
)
print(
    f"\nDIRECTION_MAGNITUDE={DIRECTION_MAGNITUDE:g}: "
    f"{mean:6.3f} +/- {sem:5.3f} bits over {count:3d} trajectories"
)

plot_mi_by_steps(direction_information_rows, "biased direction")

estimating biased-direction trajectory: 100%|██████████| 60/60 [00:59<00:00,  1.01traj/s]


Biased-direction endpoint information summary
step size:                          500 m = 7.848e-05 rad
prior entropy:                   14.651 bits
mean H(end | trajectory):         0.000 bits
median H(end | trajectory):     0.000 bits
mean I(end; trajectory):          14.651 bits
median I(end; trajectory):      14.651 bits
median endpoint MAP error:        0.2775 rad

DIRECTION_MAGNITUDE=0.75: 14.651 +/- 0.000 bits over  60 trajectories


## 4. Transformer endpoint density estimator

This section trains a conditional density model for `p(e | trajectory)`. Each model sees one fixed trajectory step count and predicts a Gaussian mixture model over the endpoint coordinates `e = (lon, y)`: mixture weights, per-component means, and per-component diagonal variances. The trajectory tokens keep the less geometry-aware `lon,y` coordinate deltas: magnetic samples, field differences, previous coordinate displacements, cumulative relative coordinate displacements, and time features. Each epoch generates a fresh synthetic training set on the fly, while feature normalization uses one fixed calibration sample. The training loss is the endpoint negative log likelihood, and the final train NLL is used as the differential entropy estimate `h(e | trajectory)`. The final plot reports `I(e; trajectory) = h(e) - h(e | trajectory)` with the continuous uniform endpoint entropy `h(e) = log(4*pi)`.


In [9]:
import torch
from IPython.display import clear_output, display
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

# Neural estimator controls. These defaults favor reducing underfit; lower them
# if a quick CPU-only pass is more important than a smooth density estimate.
NEURAL_TRAIN_SAMPLES = 8_192
NEURAL_NORMALIZATION_SAMPLES = NEURAL_TRAIN_SAMPLES
NEURAL_EVAL_SAMPLES = NEURAL_TRAIN_SAMPLES
NEURAL_EPOCHS = 120
NEURAL_BATCH_SIZE = 512
NEURAL_LEARNING_RATE = 1e-3
NEURAL_WEIGHT_DECAY = 5e-5
NEURAL_HIDDEN_DIM = 128
NEURAL_ATTENTION_HEADS = 8
NEURAL_TRANSFORMER_LAYERS = 4
NEURAL_DROPOUT = 0.02
NEURAL_GMM_COMPONENTS = 16
NEURAL_REALTIME_PLOT_EVERY = 5
NEURAL_SEED = SEED + 10_000

# The model still predicts endpoint coordinates as (lon, y), but meter-scale uncertainty
# is tiny in angular coordinates. The GMM variance head must be allowed to get much
# sharper than the older kilometer-scale default.
NEURAL_MIN_ENDPOINT_STD_M = 0.05
NEURAL_MIN_ENDPOINT_STD_COORD = NEURAL_MIN_ENDPOINT_STD_M / EARTH_RADIUS_M
NEURAL_MAX_ENDPOINT_STD_COORD = 2.0 * math.pi
NEURAL_LOG_VARIANCE_MIN = 2.0 * math.log(NEURAL_MIN_ENDPOINT_STD_COORD)
NEURAL_LOG_VARIANCE_MAX = 2.0 * math.log(NEURAL_MAX_ENDPOINT_STD_COORD)
NEURAL_INITIAL_LOG_VARIANCE = 0.0
NEURAL_LOG_VARIANCE_INITIAL_BIAS = math.log(
    (NEURAL_INITIAL_LOG_VARIANCE - NEURAL_LOG_VARIANCE_MIN)
    / (NEURAL_LOG_VARIANCE_MAX - NEURAL_INITIAL_LOG_VARIANCE)
)
NEURAL_FEATURE_STD_FLOOR = 1e-12

ENDPOINT_CONTINUOUS_ENTROPY_NATS = math.log(4.0 * math.pi)
ENDPOINT_CONTINUOUS_ENTROPY_BITS = ENDPOINT_CONTINUOUS_ENTROPY_NATS / math.log(2.0)

NEURAL_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(NEURAL_SEED)
print(f"neural endpoint device: {NEURAL_DEVICE}")
print(
    "continuous endpoint prior entropy: "
    f"{ENDPOINT_CONTINUOUS_ENTROPY_NATS:.3f} nats = "
    f"{ENDPOINT_CONTINUOUS_ENTROPY_BITS:.3f} bits"
)


def apply_walk_batch(start_lon, start_y, step_size, headings):
    lon = np.asarray(start_lon, dtype=float)
    y = np.asarray(start_y, dtype=float)
    step_size = np.asarray(step_size, dtype=float)
    step_scale = step_size[:, np.newaxis] if step_size.ndim else step_size
    lons = [lon]
    ys = [y]

    for step_index in range(headings.shape[1]):
        r = sphere_xyz(lon, y)
        east, north = local_east_north(lon, y)
        heading = headings[:, step_index]
        tangent = (
            np.cos(heading)[:, np.newaxis] * east
            + np.sin(heading)[:, np.newaxis] * north
        )
        r_next = np.cos(step_scale) * r + np.sin(step_scale) * tangent
        r_next = r_next / np.linalg.norm(r_next, axis=-1, keepdims=True)
        lon, y = lon_y_from_xyz(r_next)
        lons.append(lon)
        ys.append(y)

    return np.stack(lons, axis=1), np.stack(ys, axis=1)


def make_neural_endpoint_dataset(rng, step_count, samples):
    start_lon = rng.uniform(-np.pi, np.pi, size=samples)
    start_y = rng.uniform(-1.0, 1.0, size=samples)
    headings = rng.vonmises(
        DIRECTION_HEADING,
        DIRECTION_MAGNITUDE * DIRECTION_MAX_KAPPA,
        size=(samples, step_count),
    )

    positions_lon, positions_y = apply_walk_batch(
        start_lon, start_y, STEP_SIZE, headings
    )
    clean_B = magnetic_field(positions_lon, positions_y)
    observed_B = clean_B + rng.normal(0.0, NOISE, size=clean_B.shape)

    previous_displacement = np.zeros((samples, step_count + 1, 2), dtype=float)
    previous_displacement[:, 1:, 0] = wrap_lon(np.diff(positions_lon, axis=1))
    previous_displacement[:, 1:, 1] = np.diff(positions_y, axis=1)
    cumulative_displacement = np.cumsum(previous_displacement, axis=1)

    field_delta = np.zeros_like(observed_B)
    field_delta[:, 1:, :] = np.diff(observed_B, axis=1)

    token_time = np.broadcast_to(
        np.linspace(0.0, 1.0, step_count + 1, dtype=float)[None, :, None],
        (samples, step_count + 1, 1),
    )
    time_phase = 2.0 * np.pi * token_time
    time_features = np.concatenate(
        [token_time, np.sin(time_phase), np.cos(time_phase)], axis=-1
    )
    features = np.concatenate(
        [
            observed_B,
            field_delta,
            previous_displacement,
            cumulative_displacement,
            time_features,
        ],
        axis=-1,
    ).astype(np.float32)
    targets = np.stack(
        [wrap_lon(positions_lon[:, -1]), positions_y[:, -1]], axis=-1
    ).astype(np.float32)
    return features, targets


def normalize_neural_features(train_features):
    mean = train_features.mean(axis=(0, 1), keepdims=True)
    std = np.maximum(
        train_features.std(axis=(0, 1), keepdims=True), NEURAL_FEATURE_STD_FLOOR
    )
    return (train_features - mean) / std, mean, std


def apply_neural_feature_normalization(features, mean, std):
    return ((features - mean) / std).astype(np.float32)

neural endpoint device: cuda
continuous endpoint prior entropy: 2.531 nats = 3.651 bits


In [10]:
class EndpointTransformer(nn.Module):
    def __init__(
        self,
        input_dim,
        sequence_length,
        hidden_dim=NEURAL_HIDDEN_DIM,
        heads=NEURAL_ATTENTION_HEADS,
        layers=NEURAL_TRANSFORMER_LAYERS,
        dropout=NEURAL_DROPOUT,
        mixture_components=NEURAL_GMM_COMPONENTS,
    ):
        super().__init__()
        self.mixture_components = mixture_components
        self.input_projection = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.cls_token = nn.Parameter(torch.zeros(1, 1, hidden_dim))
        self.position_embedding = nn.Parameter(
            torch.zeros(1, sequence_length + 1, hidden_dim)
        )
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=heads,
            dim_feedforward=4 * hidden_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=layers)
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, 2 * hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(2 * hidden_dim, 2 * hidden_dim),
            nn.GELU(),
            nn.Linear(2 * hidden_dim, 5 * mixture_components),
        )

    def forward(self, trajectory_features):
        tokens = self.input_projection(trajectory_features)
        cls = self.cls_token.expand(tokens.shape[0], -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)
        tokens = tokens + self.position_embedding[:, : tokens.shape[1], :]
        pooled = self.encoder(tokens)[:, 0]
        raw = self.head(pooled)

        components = self.mixture_components
        logits = raw[:, :components]
        raw_means = raw[:, components : components + 2 * components].reshape(
            -1, components, 2
        )
        raw_log_variance = raw[:, components + 2 * components :].reshape(
            -1, components, 2
        )
        means = torch.stack(
            [
                math.pi * torch.tanh(raw_means[..., 0]),
                torch.tanh(raw_means[..., 1]),
            ],
            dim=-1,
        )
        variance_unit = torch.sigmoid(
            raw_log_variance + NEURAL_LOG_VARIANCE_INITIAL_BIAS
        )
        log_variance = (
            NEURAL_LOG_VARIANCE_MIN
            + (NEURAL_LOG_VARIANCE_MAX - NEURAL_LOG_VARIANCE_MIN) * variance_unit
        )
        return {
            "logits": logits,
            "means": means,
            "log_variance": log_variance,
        }


def wrapped_lon_residual(target_lon, mean_lon):
    return torch.remainder(target_lon - mean_lon + math.pi, 2.0 * math.pi) - math.pi


def endpoint_gmm_nll(prediction, target, reduction="mean"):
    logits = prediction["logits"]
    means = prediction["means"]
    log_variance = prediction["log_variance"]
    residual = torch.stack(
        [
            wrapped_lon_residual(target[:, None, 0], means[..., 0]),
            target[:, None, 1] - means[..., 1],
        ],
        dim=-1,
    )
    component_log_prob = -0.5 * (
        torch.sum(residual * residual / torch.exp(log_variance), dim=-1)
        + torch.sum(log_variance, dim=-1)
        + 2.0 * math.log(2.0 * math.pi)
    )
    log_prob = torch.logsumexp(
        torch.log_softmax(logits, dim=-1) + component_log_prob,
        dim=-1,
    )
    nll = -log_prob
    if reduction == "none":
        return nll
    if reduction == "sum":
        return nll.sum()
    return nll.mean()


def gmm_predictive_mean(prediction):
    weights = torch.softmax(prediction["logits"], dim=-1)
    means = prediction["means"]
    lon = torch.atan2(
        torch.sum(weights * torch.sin(means[..., 0]), dim=-1),
        torch.sum(weights * torch.cos(means[..., 0]), dim=-1),
    )
    y = torch.sum(weights * means[..., 1], dim=-1)
    return torch.stack([lon, y], dim=-1)


def evaluate_endpoint_transformer(
    model, features, targets, batch_size=NEURAL_BATCH_SIZE
):
    model.eval()
    nll_values = []
    predictions = []
    loader = DataLoader(
        TensorDataset(torch.as_tensor(features), torch.as_tensor(targets)),
        batch_size=batch_size,
        shuffle=False,
    )
    with torch.no_grad():
        for batch_features, batch_targets in loader:
            batch_features = batch_features.to(NEURAL_DEVICE)
            batch_targets = batch_targets.to(NEURAL_DEVICE)
            prediction = model(batch_features)
            nll_values.append(
                endpoint_gmm_nll(prediction, batch_targets, reduction="none")
                .detach()
                .cpu()
            )
            predictions.append(gmm_predictive_mean(prediction).detach().cpu())

    nll_values = torch.cat(nll_values).numpy()
    predictions = torch.cat(predictions).numpy()
    target_np = np.asarray(targets)
    mean_error = angular_distance(
        target_np[:, 0], target_np[:, 1], predictions[:, 0], predictions[:, 1]
    ).mean()
    sem = nll_values.std(ddof=1) / math.sqrt(nll_values.size)
    return {
        "nll_nats": float(nll_values.mean()),
        "nll_sem_nats": float(sem),
        "nll_bits": float(nll_values.mean() / math.log(2.0)),
        "nll_sem_bits": float(sem / math.log(2.0)),
        "mean_endpoint_error_rad": float(mean_error),
    }


def train_endpoint_transformer_for_steps(step_count, rng):
    normalization_features, _ = make_neural_endpoint_dataset(
        rng, step_count, NEURAL_NORMALIZATION_SAMPLES
    )
    normalized_sample, feature_mean, feature_std = normalize_neural_features(
        normalization_features
    )

    model = EndpointTransformer(
        input_dim=normalized_sample.shape[-1],
        sequence_length=normalized_sample.shape[1],
    ).to(NEURAL_DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=NEURAL_LEARNING_RATE,
        weight_decay=NEURAL_WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=NEURAL_EPOCHS,
        eta_min=0.05 * NEURAL_LEARNING_RATE,
    )
    history = []
    final_train_features = None
    final_train_targets = None
    epoch_iterator = tqdm(
        range(1, NEURAL_EPOCHS + 1),
        desc=f"steps={step_count}",
        leave=False,
        unit="epoch",
    )
    for epoch in epoch_iterator:
        train_features, train_targets = make_neural_endpoint_dataset(
            rng, step_count, NEURAL_TRAIN_SAMPLES
        )
        train_features = apply_neural_feature_normalization(
            train_features, feature_mean, feature_std
        )
        final_train_features = train_features
        final_train_targets = train_targets
        train_loader = DataLoader(
            TensorDataset(
                torch.as_tensor(train_features), torch.as_tensor(train_targets)
            ),
            batch_size=NEURAL_BATCH_SIZE,
            shuffle=True,
        )
        model.train()
        train_losses = []
        for batch_features, batch_targets in train_loader:
            batch_features = batch_features.to(NEURAL_DEVICE)
            batch_targets = batch_targets.to(NEURAL_DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = endpoint_gmm_nll(model(batch_features), batch_targets)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_losses.append(float(loss.detach().cpu()))

        row = {
            "epoch": epoch,
            "train_nll_nats": float(np.mean(train_losses)),
            "train_nll_bits": float(np.mean(train_losses) / math.log(2.0)),
        }
        history.append(row)
        scheduler.step()
        epoch_iterator.set_postfix(train=f"{row['train_nll_nats']:.3f}")

        if (
            epoch == 1
            or epoch % NEURAL_REALTIME_PLOT_EVERY == 0
            or epoch == NEURAL_EPOCHS
        ):
            clear_output(wait=True)
            fig = go.Figure()
            fig.add_trace(
                go.Scatter(
                    x=[item["epoch"] for item in history],
                    y=[item["train_nll_bits"] for item in history],
                    mode="lines+markers",
                    name=f"steps={step_count}",
                )
            )
            fig.update_xaxes(title_text="epoch")
            fig.update_yaxes(title_text="train NLL = h(e | trajectory) estimate [bits]")
            fig.update_layout(
                title=f"Training Transformer GMM endpoint density, steps={step_count}, epoch {epoch}/{NEURAL_EPOCHS}",
                height=420,
                width=900,
                margin={"l": 80, "r": 35, "t": 70, "b": 70},
            )
            display(fig)

    eval_features, eval_targets = make_neural_endpoint_dataset(
        rng, step_count, NEURAL_EVAL_SAMPLES
    )
    eval_features = apply_neural_feature_normalization(
        eval_features, feature_mean, feature_std
    )
    final_metrics = evaluate_endpoint_transformer(model, eval_features, eval_targets)
    return {
        "step_count": step_count,
        "model": model,
        "history": history,
        "feature_mean": feature_mean,
        "feature_std": feature_std,
        "final_metrics": final_metrics,
    }

In [11]:
neural_rng = np.random.default_rng(NEURAL_SEED)
endpoint_transformer_results = {}
endpoint_transformer_rows = []

for step_count in tqdm(STEP_COUNT_CHOICES, desc="training endpoint transformers"):
    result = train_endpoint_transformer_for_steps(int(step_count), neural_rng)
    final = result["final_metrics"]
    mi_nats = ENDPOINT_CONTINUOUS_ENTROPY_NATS - final["nll_nats"]
    endpoint_transformer_results[int(step_count)] = result
    endpoint_transformer_rows.append(
        {
            "steps": int(step_count),
            "conditional_entropy_nats": final["nll_nats"],
            "conditional_entropy_sem_nats": final["nll_sem_nats"],
            "conditional_entropy_bits": final["nll_bits"],
            "conditional_entropy_sem_bits": final["nll_sem_bits"],
            "end_mutual_information_nats": mi_nats,
            "end_mutual_information_bits": mi_nats / math.log(2.0),
            "end_mutual_information_sem_bits": final["nll_sem_bits"],
            "mean_endpoint_error_rad": final["mean_endpoint_error_rad"],
        }
    )

print("\nTransformer endpoint density summary")
print(f"h(e): {ENDPOINT_CONTINUOUS_ENTROPY_BITS:.3f} bits")
for row in endpoint_transformer_rows:
    print(
        f"steps={row['steps']:2d}  "
        f"h(e|traj)={row['conditional_entropy_bits']:7.3f} +/- "
        f"{row['conditional_entropy_sem_bits']:.3f} bits  "
        f"I={row['end_mutual_information_bits']:7.3f} bits  "
        f"mean endpoint error={row['mean_endpoint_error_rad']:.4f} rad"
    )

training endpoint transformers: 100%|██████████| 6/6 [50:14<00:00, 502.37s/it]


Transformer endpoint density summary
h(e): 3.651 bits
steps= 1  h(e|traj)= -3.942 +/- 0.022 bits  I=  7.594 bits  mean endpoint error=0.2456 rad
steps= 2  h(e|traj)= -4.034 +/- 0.020 bits  I=  7.685 bits  mean endpoint error=0.2296 rad
steps= 4  h(e|traj)= -4.080 +/- 0.022 bits  I=  7.732 bits  mean endpoint error=0.2196 rad
steps= 8  h(e|traj)= -4.197 +/- 0.021 bits  I=  7.849 bits  mean endpoint error=0.1949 rad
steps=12  h(e|traj)= -4.398 +/- 0.022 bits  I=  8.049 bits  mean endpoint error=0.1846 rad
steps=16  h(e|traj)= -4.559 +/- 0.023 bits  I=  8.211 bits  mean endpoint error=0.1703 rad


In [12]:
def plot_transformer_training_curves(results):
    fig = go.Figure()
    for step_count, result in results.items():
        history = result["history"]
        fig.add_trace(
            go.Scatter(
                x=[row["epoch"] for row in history],
                y=[row["train_nll_nats"] / math.log(2.0) for row in history],
                mode="lines",
                name=f"steps={step_count}",
            )
        )
    fig.update_xaxes(title_text="epoch")
    fig.update_yaxes(title_text="train NLL = h(e | trajectory) estimate [bits]")
    fig.update_layout(
        title="Transformer GMM endpoint density training loss",
        height=470,
        width=900,
        margin={"l": 80, "r": 35, "t": 70, "b": 70},
    )
    fig.show()


def plot_transformer_mi_by_steps(rows):
    rows = sorted(rows, key=lambda row: row["steps"])
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=[row["steps"] for row in rows],
            y=[row["end_mutual_information_bits"] for row in rows],
            error_y={
                "type": "data",
                "array": [row["end_mutual_information_sem_bits"] for row in rows],
                "visible": True,
            },
            mode="lines+markers+text",
            textposition="top center",
            name="Transformer GMM NLL estimate",
        )
    )
    fig.add_hline(y=0.0, line_color="gray", line_dash="dot", line_width=1.0)
    fig.update_xaxes(title_text="steps count", dtick=1)
    fig.update_yaxes(title_text="I(e; trajectory) = h(e) - final train NLL [bits]")
    fig.update_layout(
        title="Transformer GMM continuous endpoint mutual information by trajectory steps",
        height=470,
        width=900,
        margin={"l": 80, "r": 35, "t": 75, "b": 75},
    )
    fig.show()


plot_transformer_training_curves(endpoint_transformer_results)
plot_transformer_mi_by_steps(endpoint_transformer_rows)

In [13]:
CROSS_STEP_EVAL_SAMPLES = NEURAL_TRAIN_SAMPLES


def endpoint_transformer_forward_variable_length(model, trajectory_features):
    tokens = model.input_projection(trajectory_features)
    cls = model.cls_token.expand(tokens.shape[0], -1, -1)
    tokens = torch.cat([cls, tokens], dim=1)

    position_embedding = model.position_embedding
    if position_embedding.shape[1] == tokens.shape[1]:
        resized_position_embedding = position_embedding
    else:
        cls_position = position_embedding[:, :1, :]
        trajectory_position = position_embedding[:, 1:, :].transpose(1, 2)
        trajectory_position = nn.functional.interpolate(
            trajectory_position,
            size=trajectory_features.shape[1],
            mode="linear",
            align_corners=True,
        ).transpose(1, 2)
        resized_position_embedding = torch.cat(
            [cls_position, trajectory_position], dim=1
        )

    pooled = model.encoder(tokens + resized_position_embedding)[:, 0]
    raw = model.head(pooled)

    components = model.mixture_components
    logits = raw[:, :components]
    raw_means = raw[:, components : components + 2 * components].reshape(
        -1, components, 2
    )
    raw_log_variance = raw[:, components + 2 * components :].reshape(-1, components, 2)
    means = torch.stack(
        [
            math.pi * torch.tanh(raw_means[..., 0]),
            torch.tanh(raw_means[..., 1]),
        ],
        dim=-1,
    )
    variance_unit = torch.sigmoid(raw_log_variance + NEURAL_LOG_VARIANCE_INITIAL_BIAS)
    log_variance = (
        NEURAL_LOG_VARIANCE_MIN
        + (NEURAL_LOG_VARIANCE_MAX - NEURAL_LOG_VARIANCE_MIN) * variance_unit
    )
    return {"logits": logits, "means": means, "log_variance": log_variance}


def evaluate_endpoint_transformer_variable_length(
    model,
    features,
    targets,
    batch_size=NEURAL_BATCH_SIZE,
):
    model.eval()
    nll_values = []
    loader = DataLoader(
        TensorDataset(torch.as_tensor(features), torch.as_tensor(targets)),
        batch_size=batch_size,
        shuffle=False,
    )
    with torch.no_grad():
        for batch_features, batch_targets in loader:
            batch_features = batch_features.to(NEURAL_DEVICE)
            batch_targets = batch_targets.to(NEURAL_DEVICE)
            prediction = endpoint_transformer_forward_variable_length(
                model, batch_features
            )
            nll_values.append(
                endpoint_gmm_nll(prediction, batch_targets, reduction="none")
                .detach()
                .cpu()
            )

    nll_values = torch.cat(nll_values).numpy()
    sem = nll_values.std(ddof=1) / math.sqrt(nll_values.size)
    return {
        "nll_nats": float(nll_values.mean()),
        "nll_sem_nats": float(sem),
        "mi_bits": float(
            (ENDPOINT_CONTINUOUS_ENTROPY_NATS - nll_values.mean()) / math.log(2.0)
        ),
        "mi_sem_bits": float(sem / math.log(2.0)),
    }


def evaluate_models_across_step_counts(
    results,
    eval_step_counts=STEP_COUNT_CHOICES,
    eval_samples=CROSS_STEP_EVAL_SAMPLES,
    seed=NEURAL_SEED + 1,
):
    eval_rng = np.random.default_rng(seed)
    eval_sets = {
        int(step_count): make_neural_endpoint_dataset(
            eval_rng, int(step_count), eval_samples
        )
        for step_count in eval_step_counts
    }

    rows = []
    for trained_steps, result in tqdm(
        sorted(results.items()), desc="cross-step model evaluation", unit="model"
    ):
        model = result["model"]
        feature_mean = result["feature_mean"]
        feature_std = result["feature_std"]
        for eval_steps, (features, targets) in eval_sets.items():
            normalized_features = (features - feature_mean) / feature_std
            metrics = evaluate_endpoint_transformer_variable_length(
                model, normalized_features, targets
            )
            rows.append(
                {
                    "trained_steps": int(trained_steps),
                    "eval_steps": int(eval_steps),
                    **metrics,
                }
            )
    return rows


def plot_cross_step_model_evaluation(rows):
    fig = go.Figure()
    for trained_steps in sorted({row["trained_steps"] for row in rows}):
        curve = sorted(
            [row for row in rows if row["trained_steps"] == trained_steps],
            key=lambda row: row["eval_steps"],
        )
        fig.add_trace(
            go.Scatter(
                x=[row["eval_steps"] for row in curve],
                y=[row["mi_bits"] for row in curve],
                error_y={
                    "type": "data",
                    "array": [row["mi_sem_bits"] for row in curve],
                    "visible": True,
                },
                mode="lines+markers",
                name=f"trained steps={trained_steps}",
                hovertemplate=(
                    "trained steps=%{customdata[0]}<br>"
                    "eval steps=%{x}<br>"
                    "I=%{y:.3f} bits<br>"
                    "NLL=%{customdata[1]:.3f} nats<extra></extra>"
                ),
                customdata=[[trained_steps, row["nll_nats"]] for row in curve],
            )
        )
    fig.add_hline(y=0.0, line_color="gray", line_dash="dot", line_width=1.0)
    fig.update_xaxes(title_text="evaluation steps count", dtick=1)
    fig.update_yaxes(title_text="I(e; trajectory) from evaluation NLL [bits]")
    fig.update_layout(
        title="Cross-step evaluation of fixed-step Transformer GMM models",
        height=520,
        width=950,
        margin={"l": 80, "r": 35, "t": 75, "b": 75},
    )
    fig.show()


cross_step_evaluation_rows = evaluate_models_across_step_counts(
    endpoint_transformer_results
)
plot_cross_step_model_evaluation(cross_step_evaluation_rows)

cross-step model evaluation: 100%|██████████| 6/6 [00:40<00:00,  6.69s/model]
